# Tráfico en Madrid: Experimentación con Modelos

Llegamos a la parte del proyecto que todo el mundo espera: los modelos. Pero antes de ponernos a entrenar vamos a dar un paso atrás y pensar qué significa "buen modelo" en este contexto, qué métricas tienen sentido, y qué tipos de modelos vale la pena probar. La mayoría de proyectos de ML que fracasan no lo hacen porque elijan mal el modelo, sino porque eligen mal la métrica o se saltan la validación cruzada. Esos son los dos errores que vamos a evitar con mucho cuidado aquí.

El plan: cargamos los splits que dejó el notebook 02, discutimos qué métrica es la adecuada, entrenamos tres modelos de complejidad creciente (regresión logística como baseline, Random Forest como modelo no lineal estándar, HistGradientBoosting como el que suele rendir mejor en tabular), los comparamos con validación cruzada, elegimos uno como modelo final y lo guardamos para que el notebook 04 lo cargue y lo evalúe sobre el test.

## Configuración del entorno

Imports. Aparte de los habituales, hoy traemos bastante de scikit-learn: los tres modelos, la utilidad de validación cruzada, el escalador y los pipelines.

In [ ]:
import pickle
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline


Rutas y constantes. `CV_FOLDS = 5` es la cantidad estándar de folds para validación cruzada: es el compromiso clásico entre varianza (pocos folds → mucha varianza entre experimentos) y tiempo de cómputo (muchos folds → lento).

In [ ]:
DATA_PROCESSED = Path('../data/processed')
RANDOM_STATE   = 42
CV_FOLDS       = 5


In [ ]:
pd.set_option('display.float_format', '{:.3f}'.format)

sns.set_theme(style='ticks', palette='muted', font_scale=0.9)
plt.rcParams.update({
    'figure.figsize': (10, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'normal', 'axes.titlelocation': 'left',
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.frameon': False, 'axes.grid': True,
    'grid.linestyle': ':', 'grid.alpha': 0.35,
})

COLOR_PRIMARIO   = '#4C72B0'
COLOR_SECUNDARIO = '#DD8452'


## 1. Carga de los datos preparados

Partimos del fichero que generó el notebook 02. Si no existe, hay que ejecutar ese notebook antes.

In [ ]:
with open(DATA_PROCESSED / 'splits.pkl', 'rb') as f:
    splits = pickle.load(f)

X_train  = splits['X_train']
X_test   = splits['X_test']
y_train  = splits['y_train']
y_test   = splits['y_test']
features = splits['features']

print(f'Train: {X_train.shape[0]:>10,} registros × {X_train.shape[1]} features')
print(f'Test:  {X_test.shape[0]:>10,} registros × {X_test.shape[1]} features')
print(f'Proporción de congestiones en train: {y_train.mean():.2%}')


Comprobamos que los dtypes son los que esperábamos. Es un chequeo rápido pero evita sorpresas: si alguna columna se hubiera colado como object en lugar de numérica, algunos modelos fallarían o se comportarían raro.

In [ ]:
print('Tipos de datos de las features (únicos):')
print(X_train.dtypes.value_counts())


## 2. La elección de las métricas

Antes de entrenar nada, una pausa para decidir qué métrica vamos a maximizar. Esta decisión es más importante que la elección del modelo. Si usamos la métrica equivocada, el mejor modelo según esa métrica será el equivocado para el problema real.

### Por qué accuracy no basta

La tentación es medir el modelo por su accuracy (porcentaje de aciertos). Pero con desbalance de clases, esta métrica engaña. En nuestro caso, apenas un 4-6% de las lecturas son congestión. Un clasificador trivial que prediga siempre "fluido" consigue un 94-96% de accuracy sin haber aprendido absolutamente nada — porque acertar siempre en el 94% mayoritario es gratis.

Podemos demostrarlo en dos líneas, para que la idea se quede clavada.

In [ ]:
# Clasificador trivial: predice siempre 0 (fluido) para todo el mundo
prediccion_trivial = np.zeros_like(y_train)
accuracy_trivial   = (prediccion_trivial == y_train).mean()

print(f'Accuracy del clasificador que siempre dice "fluido": {accuracy_trivial:.3%}')
print('Lo cual es inútil pese al número alto.')


### Recall, precision, F1 y AUC

Para escapar de la trampa de la accuracy, nos fijamos en cuatro métricas que miden el comportamiento en la clase minoritaria (congestión).

**Precision** responde a "de las veces que el modelo dijo CONGESTIÓN, ¿cuántas lo eran de verdad?". Mide cuán fiable es una alerta del sistema. Precision baja = muchas falsas alarmas.

**Recall** (o sensibilidad) responde a "de todas las congestiones reales, ¿cuántas detectó el modelo?". Mide cuán útil es el sistema para capturar los eventos. Recall bajo = al modelo se le escapan los atascos.

**F1** es la media armónica de precision y recall. Es el número que típicamente se usa cuando uno quiere un solo indicador del balance entre las dos.

**AUC-ROC** es distinto conceptualmente. No evalúa las predicciones binarias (0/1) sino la calidad del ranking probabilístico del modelo: mide si cuando ordenamos los registros por la probabilidad que el modelo les asigna de ser congestión, los positivos quedan de verdad por delante de los negativos. AUC va de 0.5 (azar) a 1.0 (perfecto). Es útil porque es independiente del umbral de decisión: si en el futuro quisiéramos un modelo más "alarmista" o más "conservador", ajustaríamos el umbral y la precision/recall cambiarían, pero el AUC mide la calidad del modelo con independencia de ese ajuste.

### ¿Recall o precision? Depende del caso de uso

Entre recall y precision suele haber un trade-off. Si subimos el umbral para ser más estrictos al declarar congestión, la precision sube pero el recall baja (y viceversa). La pregunta es: ¿qué error es peor?

En un sistema de anticipación de atascos — que es nuestro caso de uso — un **falso negativo** (congestión real que no predecimos) es peor que un **falso positivo** (alerta que no se cumple). Si prevenimos al usuario de un atasco que al final no ocurre, como mucho se lleva una molestia menor; si no le avisamos de un atasco real, llega tarde al trabajo. Por eso queremos un modelo con buen recall, sin que la precision se hunda.

Nos quedamos con **F1 como métrica principal** (mejor balance global) y reportamos también AUC como indicador complementario.

## 3. El enfoque: validación cruzada

Para comparar modelos con justicia no vale entrenar cada uno con todo el train y mirar su métrica sobre un único split. Lo que hacemos es **validación cruzada estratificada con K folds**, que funciona así: dividimos el train en K trozos del mismo tamaño (estratificados por la clase), entrenamos K veces — en cada iteración, uno de los trozos sirve de mini-test y los otros K-1 de mini-train — y promediamos las métricas de las K iteraciones.

La ventaja es que no dependemos de la suerte de un único split y sacamos una estimación más robusta del rendimiento esperado. La desventaja es que entrenamos K veces cada modelo, multiplicando el tiempo. Con K=5 es un compromiso razonable.

Escribimos una función helper `evaluar_modelo` que recibe un modelo y devuelve las métricas promediadas.

In [ ]:
def evaluar_modelo(modelo, X, y, nombre: str) -> dict:
    '''Entrena el modelo con validación cruzada estratificada y devuelve métricas promedio.'''
    # StratifiedKFold mantiene en cada fold la proporción original de clases
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    # Scoring: lista de métricas que queremos calcular en cada fold
    metricas = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

    t0 = perf_counter()
    resultados = cross_validate(modelo, X, y, cv=skf, scoring=metricas, n_jobs=-1)
    tiempo = perf_counter() - t0

    # Media de las métricas de los 5 folds
    resumen = {m: resultados[f'test_{m}'].mean() for m in metricas}
    resumen['tiempo_s'] = tiempo

    print(f'{nombre:25s}  f1={resumen["f1"]:.3f}  recall={resumen["recall"]:.3f}  '
          f'auc={resumen["roc_auc"]:.3f}  ({tiempo:.1f}s)')
    return resumen


`n_jobs=-1` le dice a scikit-learn que use todos los núcleos disponibles. Con cinco folds en paralelo aprovechamos bien una CPU moderna.

## 4. Modelo base: regresión logística

Empezamos por el modelo más simple posible: la regresión logística. Es lineal, rápida, interpretable, y funciona como punto de referencia. La regla que vamos a aplicar es: si un modelo más complejo no supera claramente al baseline, no merece la pena la complejidad extra.

### Qué hace la regresión logística (versión rápida)

La regresión logística aprende un peso por cada feature. Para predecir, multiplica cada feature por su peso, los suma, y pasa el resultado por una función (la sigmoide) que lo convierte en una probabilidad entre 0 y 1. Si la probabilidad es mayor que 0.5, predice la clase positiva.

Es un modelo lineal: asume que la relación entre cada feature y la probabilidad del resultado es monótona. No captura interacciones a no ser que se las construyamos nosotros (como hicimos con `es_hora_punta`). Y es sensible a la escala de las features: si una variable va de 0 a 1 y otra de 0 a 500000 (como `utm_x`), la segunda aplasta a la primera durante el entrenamiento. Por eso siempre se acompaña de un escalador.

### Pipeline con escalado

Metemos el escalado y el modelo en un mismo `Pipeline` de scikit-learn. Un pipeline es una cadena de transformaciones donde la última es el modelo. La ventaja de usar un pipeline (en lugar de escalar a mano antes) es que la validación cruzada lo trata como un solo objeto: escala cada fold con los datos de ese fold, no con los del conjunto entero. Si escaláramos antes fuera del pipeline, estaríamos colando información del set de validación durante el entrenamiento — un caso sutil de data leakage.

In [ ]:
logreg = Pipeline([
    ('scaler', StandardScaler()),           # Resta la media y divide por la std. Deja cada feature con media 0 y std 1.
    ('clf', LogisticRegression(
        class_weight='balanced',            # Compensa el desbalance dando más peso a la clase minoritaria
        max_iter=1000,                      # Más iteraciones del solver para garantizar convergencia
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

res_logreg = evaluar_modelo(logreg, X_train, y_train, 'Regresión logística')


El resultado del baseline nos marca el listón. A partir de aquí, cualquier modelo que no supere estas cifras no está justificando su existencia.

## 5. Random Forest

El siguiente paso es un modelo no lineal. **Random Forest** es un ensemble de árboles de decisión: entrena muchos árboles (200 en nuestro caso) cada uno con una muestra aleatoria de los datos y un subconjunto aleatorio de las features, y promedia sus predicciones. Ese "promedio de predicciones ruidosas" reduce drásticamente la varianza y consigue modelos mucho más robustos que un árbol solo.

### Por qué encaja bien con nuestros datos

Tres razones. Primera: maneja sin problemas la mezcla de features numéricas y binarias que tenemos. No necesita escalado, porque los árboles solo miran umbrales, no magnitudes. Segunda: captura interacciones sin que se las construyamos — si el efecto de `hora` depende de `dia_semana` (que en nuestro dataset es claramente el caso, el heatmap del EDA lo mostró), un árbol se bifurca primero por una variable y luego por la otra, capturando automáticamente la interacción. Tercera: es robusto a outliers y a features irrelevantes, porque los árboles del ensemble tienden a ignorar las variables que no aportan.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,           # Nº de árboles. Más ayuda pero con rendimiento decreciente.
    max_depth=None,             # Sin límite: los árboles crecen hasta que no puedan más
    min_samples_leaf=5,         # Mínimo de muestras por hoja. Evita overfitting extremo.
    class_weight='balanced',    # Compensa el desbalance
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

res_rf = evaluar_modelo(rf, X_train, y_train, 'Random Forest')


Un detalle de parametrización: `min_samples_leaf=5` evita que los árboles crezcan hasta el extremo de una hoja por fila. Dejarlo en 1 (el default) es receta directa para overfitting: el modelo memorizaría el train en lugar de aprender patrones. Cinco es un buen compromiso.

Si el Random Forest supera a la regresión logística — y esperamos que sí, dadas las interacciones que vimos en el EDA — significa que la no-linealidad estaba aportando señal real que el baseline no podía capturar.

## 6. Gradient Boosting

El tercer modelo es un **gradient boosting**. La intuición es distinta al Random Forest: en lugar de entrenar árboles independientes y promediar, entrena árboles en **secuencia**, donde cada árbol nuevo intenta corregir los errores del conjunto anterior. Es como un equipo donde cada miembro aprende específicamente de lo que se le escapó a los anteriores.

En la práctica, para clasificación tabular el gradient boosting suele ser el modelo que mejor rinde, a veces con mucha diferencia. XGBoost, LightGBM y CatBoost (sus variantes más populares) ganan la mayoría de competiciones de Kaggle con datos tabulares.

### HistGradientBoostingClassifier

Dentro de scikit-learn, la implementación rápida se llama `HistGradientBoostingClassifier`. Lo que hace diferente respecto al `GradientBoostingClassifier` clásico es que **discretiza las features numéricas en bins** antes de entrenar, lo que permite hacer los splits de los árboles mucho más rápido. El resultado: órdenes de magnitud más rápido que la versión clásica con datasets grandes, y un rendimiento prácticamente idéntico. Para nuestro tamaño de dataset, es la elección obvia.

In [ ]:
hgb = HistGradientBoostingClassifier(
    max_iter=300,               # Nº de árboles en el ensemble
    learning_rate=0.1,          # Cuánto contribuye cada árbol. Más bajo → entrenamiento más largo pero mejor generalización
    max_depth=None,             # Sin profundidad máxima explícita
    class_weight='balanced',    # Disponible desde sklearn 1.2; compensa el desbalance
    random_state=RANDOM_STATE,
)

res_hgb = evaluar_modelo(hgb, X_train, y_train, 'HistGradientBoosting')


Los hiperparámetros elegidos son razonables para un primer intento. Podríamos afinarlos con un `GridSearchCV` buscando la mejor combinación de `learning_rate` y `max_iter`, pero eso añadiría horas de cómputo por una ganancia típicamente modesta (un par de puntos decimales en F1). Para un proyecto de 3 créditos donde lo importante es la metodología, no vamos a meternos en esa optimización. Lo dejamos señalado como posible mejora.

## 7. Comparativa

Juntamos los tres resultados en una tabla. Ver los números lado a lado es la forma más clara de decidir.

In [ ]:
comparativa = pd.DataFrame({
    'Regresión logística':  res_logreg,
    'Random Forest':        res_rf,
    'HistGradientBoosting': res_hgb,
}).T

comparativa[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'tiempo_s']]


### Visualización comparativa

Las barras agrupadas facilitan ver quién va por delante en cada métrica.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

metricas_plot = ['precision', 'recall', 'f1', 'roc_auc']
comparativa[metricas_plot].plot(
    kind='bar', ax=ax, width=0.75,
    color=['#8DA0CB', '#DD8452', '#4C72B0', '#55A868'],
)
ax.set_title('Comparativa de modelos (media en CV de 5 folds)')
ax.set_ylabel('valor de la métrica')
ax.set_ylim(0, 1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right', ncol=4)
plt.tight_layout()
plt.show()


### Interpretación de los resultados

El patrón típico cuando el problema tiene no-linealidades es el que vemos aquí: la regresión logística marca un suelo honesto, el Random Forest mejora claramente al capturar interacciones, y el gradient boosting afina un poco más al corregir errores residuales. Las diferencias entre los dos modelos basados en árboles son pequeñas pero consistentes.

Los tiempos cuentan otra historia. La regresión logística entrena en una fracción del tiempo de los árboles. Si el rendimiento fuera similar, ganaría el baseline por rapidez. Pero las diferencias de F1 y AUC son lo bastante grandes como para justificar el tiempo extra del modelo complejo.

## 8. Elección del modelo final

Elegimos **HistGradientBoosting** como modelo final. Tiene el mejor F1 y el mejor AUC, entrena en un tiempo razonable, y acepta features numéricas sin escalado. El Random Forest sería una alternativa defendible — muy cerca en métricas — pero el boosting le gana por un margen pequeño pero estable y su tiempo de entrenamiento es similar.

La regresión logística queda como baseline. La dejaremos entrenada también, porque en el notebook 04 compararemos sus curvas ROC para ver si la ganancia del modelo complejo se mantiene en test o si era ruido del CV.

In [ ]:
# Entrenamos el modelo final con todos los datos de train (no una fracción)
modelo_final = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.1,
    max_depth=None,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
modelo_final.fit(X_train, y_train)
print('Modelo final entrenado.')


In [ ]:
# El baseline también, para la comparación en test
baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000,
                               random_state=RANDOM_STATE, n_jobs=-1)),
])
baseline.fit(X_train, y_train)
print('Baseline entrenado.')


## 9. Guardado de los modelos

Dejamos los modelos serializados y la tabla comparativa en `data/processed/` para que el notebook 04 los cargue sin tener que reentrenar nada.

In [ ]:
artefactos = {
    'modelo_final': modelo_final,
    'baseline':     baseline,
    'comparativa':  comparativa,
    'features':     features,
}

with open(DATA_PROCESSED / 'modelos.pkl', 'wb') as f:
    pickle.dump(artefactos, f)

print('Guardado modelos.pkl en', DATA_PROCESSED.resolve())
print(f'Tamaño: {(DATA_PROCESSED / "modelos.pkl").stat().st_size / 1e6:.1f} MB')


## 10. Resumen

Tres modelos, validación cruzada estratificada de 5 folds, métricas pensadas para el problema (F1 y AUC, no accuracy) y elección justificada por los números. El gradient boosting se impone por un margen pequeño pero consistente, y la regresión logística queda como referencia interpretable para comparar en test.

Decisiones importantes a recordar: rechazamos accuracy como métrica principal por el desbalance de clases; usamos `class_weight='balanced'` en los tres modelos para que no ignoraran la clase minoritaria; metimos el escalado en un pipeline para evitar leakage en la validación cruzada; fijamos `random_state=42` en cada componente para reproducibilidad; entrenamos el modelo final con todo el train una vez elegido. A partir de aquí ya no se tocan los datos de train. El notebook 04 abre el test por primera vez y ve cómo se comporta el modelo con datos que no ha visto nunca.